### Data Transformation

In [2]:
# import libraries
import numpy as np
import pandas as pd

In [3]:
# load the cleaned data
df = pd.read_csv("../data/stars_clean.csv")
SKEWED = ["Temperature", "L", "R"]

In [4]:
# apply the selected transformation (log10)
for c in SKEWED:
    df[f"log_{c}"] = np.log10(df[c])

pd.DataFrame({"before": df[SKEWED].skew().values,
              "after": df[[f"log_{c}" for c in SKEWED]].skew().values},
             index=SKEWED).round(2)

,before,after
Temperature,1.32,0.44
L,2.07,0.19
R,1.95,0.36


In [ ]:
# Encode Spectral_Class as ordinal
spectral_order = {"O": 0, "B": 1, "A": 2, "F": 3, "G": 4, "K": 5, "M": 6}   # hot -> cool
df["Spectral_Ordinal"] = df["Spectral_Class"].map(spectral_order)
assert df["Spectral_Ordinal"].notna().all()

In [6]:
# One-hot encode Color
color_dummies = pd.get_dummies(df["Color"], prefix="color", dtype=int)
df = pd.concat([df, color_dummies], axis=1)
print(list(color_dummies.columns))

['color_blue', 'color_blue white', 'color_orange', 'color_orange red', 'color_pale yellow orange', 'color_red', 'color_white', 'color_white yellow', 'color_whitish', 'color_yellow white', 'color_yellowish', 'color_yellowish white']


In [7]:
# assemble the model
feature_cols = (["log_Temperature", "log_L", "log_R", "A_M", "Spectral_Ordinal"]
                + list(color_dummies.columns))

model_df = df[feature_cols + ["Type", "Type_Name"]]
print("Features:", len(feature_cols), "| rows:", len(model_df))

Features: 17 | rows: 240


In [9]:
model_df

,log_Temperature,log_L,log_R,A_M,Spectral_Ordinal,color_blue,color_blue white,color_orange,color_orange red,color_pale yellow orange,color_red,color_white,color_white yellow,color_whitish,color_yellow white,color_yellowish,color_yellowish white,Type,Type_Name
0,3.486855,-2.619789,-0.769551,16.12,6,0,0,0,0,0,1,0,0,0,0,0,0,0,Red Dwarf
1,3.483159,-3.301030,-0.811916,16.60,6,0,0,0,0,0,1,0,0,0,0,0,0,0,Red Dwarf
2,3.414973,-3.522879,-0.991400,18.70,6,0,0,0,0,0,1,0,0,0,0,0,0,0,Red Dwarf
3,3.447158,-3.698970,-0.795880,16.65,6,0,0,0,0,0,1,0,0,0,0,0,0,0,Red Dwarf
4,3.287578,-3.860121,-0.987163,20.06,6,0,0,0,0,0,1,0,0,0,0,0,0,0,Red Dwarf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,4.590396,5.573834,3.132260,-9.93,0,1,0,0,0,0,0,0,0,0,0,0,0,5,Hyper Giant
236,4.489100,5.921188,3.077004,-10.63,0,1,0,0,0,0,0,0,0,0,0,0,0,5,Hyper Giant
237,3.945912,5.730373,3.153205,-10.73,2,0,0,0,0,0,0,1,0,0,0,0,0,5,Hyper Giant
238,3.965437,5.607391,3.046105,-11.23,2,0,0,0,0,0,0,1,0,0,0,0,0,5,Hyper Giant


In [8]:
# save the transformed data
model_df.to_csv("../data/stars_transformed.csv", index=False)